In [ ]:
# 
#
# PURPOSE:
# 1. Load all final artifacts:
#    - The trained Classifier (from Notebook 02)
#    - The trained Autoencoder (from Notebook 03)
#    - The Test Data (from Notebook 02)
#    - The Classifier Threshold (from Notebook 02)
#    - The Anomaly Threshold (from Notebook 03)
# 2. Run the Test Data through *both* models to get two scores:
#    - Classifier Probability (P(Faulty))
#    - Reconstruction Error (Anomaly Score)
# 3. Create the "Hybrid UQ Plot" to analyze the results.
# 4. Save the final analysis DataFrame as a CSV.

# --- Cell 1: Setup & Imports ---
import os, json, pickle, random, pathlib
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import Dropout # Needed for MCDropout
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

print(f"TensorFlow Version: {tf.__version__}")

In [ ]:
# --- Cell 2: Constants & File Paths ---
# All I/O points to our central project directory
ARTIFACT_DIR = Path("Final_Run_Outputs")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# Input files (from Notebooks 02 & 03)
DATA_NPZ = ARTIFACT_DIR / "classification_pipeline_data.npz"
CLASSIFIER_MODEL_FILE = ARTIFACT_DIR / "classifier_mc_inference.keras"
CLASSIFIER_METRICS_FILE = ARTIFACT_DIR / "final_test_metrics.json"
ANOMALY_MODEL_FILE = ARTIFACT_DIR / "anomaly_autoencoder.keras"
ANOMALY_METRICS_FILE = ARTIFACT_DIR / "anomaly_threshold.json"

# Output files for this notebook
HYBRID_RESULTS_CSV = ARTIFACT_DIR / "hybrid_analysis_results.csv"
HYBRID_RESULTS_NPZ = ARTIFACT_DIR / "hybrid_analysis_data.npz"

# Model constants
MAX_SEQ_LEN = 2048
N_CHANNELS = 23
SEED = 1337

# --- Seeding ---
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

print(f"Loading all artifacts from: {ARTIFACT_DIR}")
print(f"Hybrid results will be saved to: {ARTIFACT_DIR}")

In [ ]:
# --- Cell 3: Define MCDropout Layer (for loading) ---
# This custom object is required to load the classifier model
class MCDropout(Dropout):
    def call(self, inputs, training=None):
        return super().call(inputs, training=True)

custom_objects = {'MCDropout': MCDropout}
print("Custom MCDropout layer defined.")

In [ ]:
# --- Cell 4: Load All Data and Models ---
print("Loading all data, models, and thresholds...")
try:
    # Load Test Data & Classifier Predictions
    data_archive = np.load(DATA_NPZ, allow_pickle=True)
    T_scaled = data_archive['T_scaled']
    y_test_true = data_archive['y_test_true'].astype(int).ravel()
    # Load predictions made by Notebook 02
    p_test_mean = data_archive['p_test_mean']
    p_test_std = data_archive['p_test_std']
    y_test_pred_classifier = data_archive['y_test_pred']
    
    # Load Models
    clf_inference = load_model(CLASSIFIER_MODEL_FILE, custom_objects=custom_objects)
    autoencoder = load_model(ANOMALY_MODEL_FILE)
    
    # Load Thresholds
    with open(CLASSIFIER_METRICS_FILE, 'r') as f:
        classifier_metrics = json.load(f)
        classifier_threshold = classifier_metrics['best_thr']
        
    with open(ANOMALY_METRICS_FILE, 'r') as f:
        anomaly_metrics = json.load(f)
        anomaly_threshold = anomaly_metrics['anomaly_threshold_mae']

    print(f"Loaded Test Data: {T_scaled.shape}")
    print("Loaded Classifier and Autoencoder models.")
    print(f"Loaded Classifier Threshold: {classifier_threshold:.4f}")
    print(f"Loaded Anomaly Threshold: {anomaly_threshold:.4f}")

except Exception as e:
    print(f"Error loading files: {e}")
    print("Please ensure '02_Classifier_and_Generation.ipynb' and '03_Anomaly_Autoencoder.ipynb' were run successfully.")

In [ ]:
# --- Cell 5: Get Reconstruction Errors ---
# The classifier predictions (prob, std) were already loaded from the NPZ.
# We just need to calculate the reconstruction errors from the autoencoder.

print(f"Calculating reconstruction errors for {len(T_scaled)} test samples...")
ae_test_errors = np.mean(
    np.abs(autoencoder.predict(T_scaled, batch_size=64) - T_scaled), 
    axis=(1, 2)
)
print("Reconstruction errors calculated.")

In [ ]:
# --- Cell 6: Create and Save Hybrid Analysis DataFrame ---
print("Creating hybrid analysis DataFrame...")
df = pd.DataFrame({
    'true_label': y_test_true,
    'pred_label_classifier': y_test_pred_classifier, # From classifier
    'class_prob': p_test_mean,
    'mc_uncertainty': p_test_std,
    'recon_error': ae_test_errors
})

# Add helper columns for analysis
df['is_correct'] = (df['true_label'] == df['pred_label_classifier'])
df['prediction_type'] = 'True Negative' # Default
df.loc[(df['true_label'] == 1) & (df['is_correct'] == True), 'prediction_type'] = 'True Positive'
df.loc[(df['true_label'] == 0) & (df['is_correct'] == False), 'prediction_type'] = 'False Positive'
df.loc[(df['true_label'] == 1) & (df['is_correct'] == False), 'prediction_type'] = 'False Negative'

# --- SAVE POINT ---
# Save the final analysis results
df.to_csv(HYBRID_RESULTS_CSV, index=False)
np.savez_compressed(
    HYBRID_RESULTS_NPZ,
    true_label=y_test_true,
    pred_label_classifier=y_test_pred_classifier,
    class_prob=p_test_mean,
    mc_uncertainty=p_test_std,
    recon_error=ae_test_errors,
    classifier_threshold=classifier_threshold,
    anomaly_threshold=anomaly_threshold
)
print(f"Hybrid analysis results saved to {HYBRID_RESULTS_CSV}")
print(f"Hybrid analysis arrays saved to {HYBRID_RESULTS_NPZ}")

print("\n--- Analysis DataFrame Head ---")
print(df.head())
print("\n--- Prediction Type Counts (from Classifier) ---")
print(df['prediction_type'].value_counts())

In [ ]:
# --- Cell 7: Plot 1 - The Hybrid UQ Scatter Plot ---
# This is your final "money plot" that proves the hybrid approach.
# It explains *why* the classifier made errors and *how* to catch them.
print("\nPlotting Hybrid UQ Analysis...")
plt.figure(figsize=(12, 8))

sns.scatterplot(
    data=df,
    x='class_prob',
    y='recon_error',
    hue='prediction_type', # Color by TP, FP, TN, FN
    style='prediction_type', # Different markers
    s=150,
    alpha=0.8
)

# Add the decision boundaries
plt.axvline(classifier_threshold, color='red', linestyle='--', 
            label=f'Classifier Threshold ({classifier_threshold:.2f})')
plt.axhline(anomaly_threshold, color='purple', linestyle='--', 
            label=f'Anomaly Threshold ({anomaly_threshold:.2f})')

plt.title('Hybrid UQ Analysis: Classifier vs. Autoencoder', fontsize=16)
plt.xlabel('Classifier Probability (P(Faulty))', fontsize=12)
plt.ylabel('Autoencoder Reconstruction Error (Anomaly Score)', fontsize=12)
plt.legend()
plt.grid(True, which='both', linestyle='--', alpha=0.5)

plt.savefig(OUT_DIR / "hybrid_uq_scatter.png")
plt.show()

In [ ]:
# --- Cell 8: Plot 2 - Classifier Prob vs. MC Uncertainty ---
# This plot shows *why* MC-Dropout failed, demonstrating that the
# "confidently wrong" errors (False Positives) have low uncertainty.
print("\nPlotting Original MC-Dropout UQ Analysis...")
plt.figure(figsize=(12, 8))

sns.scatterplot(
    data=df,
    x='class_prob',
    y='mc_uncertainty',
    hue='prediction_type',
    style='prediction_type',
    s=150,
    alpha=0.8
)

plt.axvline(classifier_threshold, color='red', linestyle='--', 
            label=f'Classifier Threshold ({classifier_threshold:.2f})')

plt.title('Original UQ Analysis: Classifier Prob vs. MC Uncertainty', fontsize=16)
plt.xlabel('Classifier Probability (P(Faulty))', fontsize=12)
plt.ylabel('MC-Dropout Uncertainty (Std. Dev.)', fontsize=12)
plt.legend()
plt.grid(True, which='both', linestyle='--', alpha=0.5)

plt.savefig(OUT_DIR / "original_uq_scatter_by_type.png")
plt.show()

print(f"\n--- Notebook 04 (Hybrid Analysis) Complete ---")
print(f"All final project artifacts are in: {OUT_DIR.resolve()}")